# 15장. 하나의 데이터 분석 프로젝트로 완성하기

이 노트북은 완료 주문 매출, 데이터 품질, 분류 모델, 선택형 외부 데이터, LLM 사용 기록, 자동화 설계와 최종 보고서를 하나의 재현 가능한 흐름으로 연결합니다.

기본 실행은 외부 네트워크를 호출하지 않습니다.


## 학습 목표

- 완료 주문만 매출에 포함합니다.
- 키 관계와 병합 후 행 수를 검증합니다.
- 고객 결과에서 원본 식별정보를 제거합니다.
- 분류 모델의 validation 선택과 test 최종 평가를 구분합니다.
- 실제 외부 파일이 있을 때만 통합합니다.
- 프로젝트 검증표와 산출물 manifest를 확인합니다.


## 1. 프로젝트 루트 설정


In [ ]:
from pathlib import Path
import sys


def find_project_root(start_path):
    start_path = Path(start_path).resolve()
    for candidate in [start_path, *start_path.parents]:
        if (
            (candidate / 'requirements.txt').exists()
            and (candidate / 'scripts').exists()
        ):
            return candidate
    raise FileNotFoundError(
        '프로젝트 루트 폴더를 찾을 수 없습니다.'
    )


PROJECT_ROOT = find_project_root(Path.cwd())

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print('프로젝트 루트:', PROJECT_ROOT)


## 2. 전체 파이프라인 실행

원본 데이터가 없다면 먼저 프로젝트 루트 터미널에서 다음 명령을 실행합니다.

```powershell
python scripts/generate_sample_data.py
```


In [ ]:
from src.final_project import run_final_project

result = run_final_project(
    PROJECT_ROOT,
    random_state=42,
)

print('최종 보고서:', result['final_report_path'])
print('산출물 manifest:', result['deliverables_path'])


## 3. 데이터 구조와 전처리 검증


In [ ]:
core = result['core']

display(core['dataset_summary'])
display(core['preprocessing_comparison'])
display(core['key_duplicate_checks'])
display(core['relationship_checks'])


## 4. 완료 주문 매출 범위와 병합 검증


In [ ]:
display(
    core['public_tables']['amount_scope_summary']
)
display(
    core['public_tables']['merge_checks']
)


모든 매출 집계는 `order_status == "completed"`인 주문만 사용합니다. 카테고리·월·고객·상품 매출 합계가 완료 주문 매출과 같아야 합니다.


In [ ]:
public_tables = core['public_tables']

totals = {
    'category': public_tables['category_sales'][
        'total_sales'
    ].sum(),
    'monthly': public_tables['monthly_sales'][
        'total_sales'
    ].sum(),
    'customer': public_tables['customer_sales'][
        'total_sales'
    ].sum(),
    'product': public_tables['product_sales'][
        'total_sales'
    ].sum(),
}
totals


## 5. 완료 주문 EDA 결과


In [ ]:
display(public_tables['category_sales'].head(10))
display(public_tables['monthly_sales'].head(12))
display(public_tables['customer_sales'].head(10))
display(public_tables['product_sales'].head(10))
display(public_tables['order_status_summary'])


고객 결과에는 `customer_id`, 이름, 이메일, 전화번호가 포함되지 않아야 합니다.


In [ ]:
public_tables['customer_sales'].columns.tolist()


## 6. 분류 모델 결과


In [ ]:
classification_result = result['classification']

display(classification_result['status'])
display(classification_result['validation_comparison'])
display(classification_result['test_metrics'])
display(classification_result['confusion_matrix'])


분류 단계는 `completed=0`, `cancelled=1`만 사용합니다. 모델과 임계값은 validation에서 선택하고 test는 최종 평가에만 사용합니다. 데이터가 부족하면 단계가 `skipped`로 기록될 수 있습니다.


## 7. 외부 데이터 통합 상태


In [ ]:
external_result = result['external']

display(external_result['status'])
display(external_result['comparison'])
display(external_result['merge_check'])


외부 통합을 실행하려면 실제 출처의 파일을 다음 위치에 준비합니다.

```text
data/external/processed/holidays.csv
```

파일이 없으면 가짜 공휴일 데이터를 만들지 않고 `reports/ch15_holidays_template.csv`를 생성합니다.


## 8. LLM 사용 로그 템플릿


In [ ]:
display(result['llm_usage_log'])


LLM을 실제로 사용한 경우에만 제공자, 모델명, 실행일, 프롬프트 버전, 검증 결과와 수정 내용을 작성합니다. 사용하지 않았다면 `미사용` 상태를 유지합니다.


## 9. 프로젝트 검증표


In [ ]:
validation = result['validation']
display(validation)


- `PASS`: 필수 기준 충족
- `WARN`: 선택 단계 미실행 또는 해석 제한
- `FAIL`: 제출 전 수정이 필요한 오류


## 10. 산출물 manifest


In [ ]:
manifest = result['manifest']
display(manifest)


## 11. 생성 파일 존재 여부 확인


In [ ]:
for name, path in result['output_paths'].items():
    print(
        name,
        'OK' if path.exists() else 'MISSING',
        path,
    )


## 12. 스크립트로 다시 실행하기

프로젝트 루트에서 다음 명령으로 같은 결과를 생성합니다.

```powershell
python scripts/run_final_project.py
```


## 정리

최종 프로젝트는 많은 기능을 넣는 작업이 아니라, 완료 주문 매출 범위·데이터 관계·모델 평가·외부 데이터 출처·개인정보·산출물 무결성을 검증하며 분석 질문에서 보고서까지 연결하는 작업입니다.
